In [1]:
import os
import shutil
import tensorflow as tf
from tensorflow.keras import layers, models
from google.colab import files

In [2]:
print("--- STEP 1: Setting up environment and downloading dataset ---")

--- STEP 1: Setting up environment and downloading dataset ---


In [3]:
for item in ['eye_disease_dataset', 'eye-diseases-classification.zip', 'eye_disease_model.h5']:
    if os.path.exists(item):
        if os.path.isdir(item): shutil.rmtree(item)
        else: os.remove(item)

In [4]:
print("Please upload your kaggle.json file:")
uploaded = files.upload()

if not uploaded:
    raise ValueError("File upload cancelled. Please upload a valid kaggle.json file to proceed.")

Please upload your kaggle.json file:


Saving kaggle.json to kaggle.json


In [5]:
uploaded_filename = list(uploaded.keys())[0]

In [6]:
kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
shutil.copy(uploaded_filename, os.path.join(kaggle_dir, "kaggle.json"))
os.chmod(os.path.join(kaggle_dir, "kaggle.json"), 0o600)

In [7]:
os.remove(uploaded_filename)

In [8]:
print("Downloading dataset from Kaggle...")
!kaggle datasets download -d gunavenkatdoddi/eye-diseases-classification
!unzip -q eye-diseases-classification.zip -d eye_disease_dataset

print("Dataset unpacked successfully.\n")

Dataset URL: https://www.kaggle.com/datasets/gunavenkatdoddi/eye-diseases-classification
License(s): ODbL-1.0
100% 736M/736M [00:04<00:00, 176MB/s]

Dataset unpacked successfully.



In [9]:
print("--- STEP 2: Building training and validation data streams ---")

DATA_DIR = "eye_disease_dataset/dataset"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(f"Verification failed: Path '{DATA_DIR}' could not be resolved.")

--- STEP 2: Building training and validation data streams ---


**Generate Training Dataset Split (80%)**

In [10]:
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

Found 4217 files belonging to 4 classes.
Using 843 files for validation.


**Generate Validation Dataset Split (20%)**

In [11]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)
class_names = train_ds.class_names

Found 4217 files belonging to 4 classes.
Using 3374 files for training.


In [12]:
# Cache data streams directly into local memory space to maximize performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

print(f"Target diagnostic classes found: {class_names}\n")

Target diagnostic classes found: ['cataract', 'diabetic_retinopathy', 'glaucoma', 'normal']



In [13]:
print("--- STEP 3: Initializing model network & starting training ---")

# Pull MobileNetV2 architecture with ImageNet feature maps
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

--- STEP 3: Initializing model network & starting training ---
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [14]:
# Build Deep Learning Functional API Graph
inputs = tf.keras.Input(shape=(224, 224, 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)  # Drop connections to prevent overfitting
outputs = layers.Dense(len(class_names), activation='softmax')(x)

model = models.Model(inputs, outputs)

# Cross-version secure compilation logic
try:
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
except ValueError:
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_with_cross_entropy',
        metrics=['accuracy']
    )

print("Training model layers on GPU hardware...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=18
)

# Save full weights and graph structure
model.save("eye_disease_model.h5")
print("Model optimized and exported as 'eye_disease_model.h5'.\n")

Training model layers on GPU hardware...
Epoch 1/18
106/106 ━━━━━━━━━━━━━━━━━━━━ 66s 314ms/step - accuracy: 0.6841 - loss: 0.7359 - val_accuracy: 0.8102 - val_loss: 0.4633
Epoch 2/18
106/106 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - accuracy: 0.8074 - loss: 0.4969 - val_accuracy: 0.8529 - val_loss: 0.4110
Epoch 3/18
106/106 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - accuracy: 0.8382 - loss: 0.4326 - val_accuracy: 0.8577 - val_loss: 0.3776
Epoch 4/18
106/106 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.8509 - loss: 0.4021 - val_accuracy: 0.8565 - val_loss: 0.3627
Epoch 5/18
106/106 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.8643 - loss: 0.3730 - val_accuracy: 0.8588 - val_loss: 0.3544
Epoch 6/18
106/106 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - accuracy: 0.8657 - loss: 0.3567 - val_accuracy: 0.8802 - val_loss: 0.3343
Epoch 7/18
106/106 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.8782 - loss: 0.3308 - val_accuracy: 0.8719 - val_loss: 0.3444
Epoch 8/18
106/106 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step -

Model optimized and exported as 'eye_disease_model.h5'.



In [15]:
# CELL A: Imports for data analysis
import matplotlib.pyplot as plt
import numpy as np
import os

print("Libraries loaded.")

Libraries loaded.


In [16]:
# ------------------------------------------------------------------------------
# STEP 4: INTERACTIVE UI LAUNCH (Gradio - Colab Optimized)
# ------------------------------------------------------------------------------
!pip install -q gradio

import gradio as gr
import numpy as np
import tensorflow as tf
from PIL import Image

def predict_eye_disease(image):
    if image is None:
        return "⚠️ No image uploaded."

    # Preprocess
    img = Image.fromarray(image.astype('uint8'), 'RGB').resize((224, 224))
    img_array = tf.keras.utils.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)

    # Predict - return only TOP result
    predictions = model.predict(img_array)[0]
    top_index = np.argmax(predictions)
    top_label = class_names[top_index]
    top_conf = predictions[top_index] * 100

    return f"🔍 Diagnosis: {top_label}  ({top_conf:.1f}% confidence)"

# ── UI Layout ──────────────────────────────────────────────────────────────────
with gr.Blocks(theme=gr.themes.Soft(), title="AI Eye Disease Detection") as demo:

    gr.Markdown("""
    # 👁️ AI Eye Disease Detection Assistant
    Upload a **retinal fundus scan** to get an instant diagnosis.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(
                label="📤 Upload Retinal Scan",
                type="numpy",
                height=300
            )
            analyze_btn = gr.Button("🔍 Analyze Image", variant="primary", size="lg")
            clear_btn = gr.Button("🗑️ Clear", size="sm")

        with gr.Column(scale=1):
            text_output = gr.Textbox(
                label="📋 Diagnosis Result",
                placeholder="Result will appear here after analysis...",
                interactive=False,
                lines=3
            )
            gr.Markdown("""
            **Possible Diagnoses:**
            - 🟢 Normal
            - 🔴 Diabetic Retinopathy
            - 🟡 Glaucoma
            - 🟠 Cataract
            """)

    analyze_btn.click(fn=predict_eye_disease, inputs=image_input, outputs=text_output)
    clear_btn.click(fn=lambda: (None, ""), inputs=[], outputs=[image_input, text_output])

print("🚀 Launching app...")
demo.launch(share=True, debug=False, quiet=True)

/tmp/ipykernel_1695/24031711.py:29: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="AI Eye Disease Detection") as demo:


🚀 Launching app...
* Running on public URL: https://e27348b4cce5dc13d0.gradio.live
